# 04 — XGBoost cost regressor

The classifier says "dent + scratch". The detector says "two boxes covering ~6% of the image". A repair shop says "$420". The cost regressor maps the model outputs to a dollar amount.

## Roadmap

1. Why **gradient boosting** instead of another neural net
2. The math: decision tree → boosting → gradient boosting
3. Features the regressor sees
4. The **calibration** trick — how we re-price without retraining
5. Runnable training on synthetic data


In [ ]:
# === Colab bootstrap ===
# Safe to re-run. On a local clone with `pip install -e .` already done this
# is a no-op; on Colab it clones the repo + installs deps the first time.
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/theDocWho/car-crash-fix-amount-predictor.git"
REPO_DIR = Path("car-crash-fix-amount-predictor")

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
if IN_COLAB:
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "-r", "requirements.txt"], check=True)

# Make `ccdp` importable whether or not the package was installed editable.
src_path = Path("src").resolve()
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

print("ccdp path:", src_path)
print("running in Colab:", IN_COLAB)


## 1. Why XGBoost here?

The input to the cost stage is *structured*: a 2048-d image feature vector plus categorical fields (make, body type, segment) plus bbox stats. For tabular structured data, **gradient-boosted trees** routinely beat neural networks — fewer hyperparameters, faster to train, less prone to overfit small datasets, and they give us *interpretable feature importances* for free.

XGBoost is the most battle-tested implementation. (`HistGradientBoostingRegressor` from scikit-learn is an equally good drop-in if you want a pure-sklearn variant — see the conversation context for that ablation idea.)

## 2. From decision trees → boosting → gradient boosting

### Decision tree
A flowchart: at each node, ask a yes/no question about one feature; descend until you hit a leaf with a prediction.

### Boosting
Train a sequence of **weak** trees. Each new tree focuses on the examples the previous trees got wrong. Sum all trees' predictions to get the final answer.

### Gradient boosting
The "focuses on what was wrong" is made precise: tree $t+1$ is fit to the **negative gradient** of the loss with respect to the current ensemble's predictions. For squared-error loss this is just the residual $y - \hat y$.

Formally, at step $t$:

$$
F_{t+1}(x) = F_t(x) + \nu \cdot h_t(x), \qquad h_t \approx -\nabla_F \mathcal{L}(y, F_t(x))
$$

where $\nu$ is the learning rate.


In [ ]:
# Visualise boosting on a 1-D toy problem.
import numpy as np, matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeRegressor

rng = np.random.default_rng(0)
x = np.linspace(0, 6, 100)
y = np.sin(x) + 0.2 * rng.standard_normal(x.shape)

X = x.reshape(-1, 1)
preds = np.zeros_like(y)
fig, axes = plt.subplots(1, 4, figsize=(15, 3.2), sharey=True)
for i, n_trees in enumerate([1, 3, 10, 50]):
    preds = np.zeros_like(y)
    for _ in range(n_trees):
        resid = y - preds
        tree = DecisionTreeRegressor(max_depth=2).fit(X, resid)
        preds += 0.3 * tree.predict(X)
    axes[i].scatter(x, y, s=8, alpha=0.4)
    axes[i].plot(x, preds, color="red")
    axes[i].set_title(f"{n_trees} trees")
plt.suptitle("Gradient boosting fits residuals tree by tree")
plt.show()


## 3. What features does the cost regressor see?

Variant B's feature row (training and inference) contains:

| Group | Features | Source |
|---|---|---|
| Image features | `f_0` … `f_2047` | ResNet50 backbone output |
| Car metadata | `year`, `make`, `body_type`, `segment` | User input or identifier model |
| BBox stats | counts, total area, max area, area-weighted class fractions | YOLOv8 detector |

The XGBoost model learns nonlinear interactions between *all* of these — e.g. "luxury sedan + scratch on door + low total area → $X" vs "old SUV + scratch on bumper + same area → $Y".

## 4. The calibration trick

Re-training XGBoost every time the catalog changes would be wasteful (and would lose us the production weights). Instead we add a **post-hoc calibrator**: a small affine transform applied on top of the XGBoost output to align it with a freshly-priced catalog.

```
cost_final = α · xgb_raw + β
```

α, β are fit by least squares on the calibration set. When you switch catalogs in the demo, only (α, β) get re-fit — the XGBoost model itself never moves. This is why the Gradio "Catalog manager" tab can re-price an image instantly.


In [ ]:
# Tiny demo: fit (alpha, beta) by closed-form least squares.
import numpy as np

raw       = np.array([100, 200, 300, 400, 500])     # XGBoost raw predictions
priced    = np.array([150, 290, 470, 580, 720])     # what the new catalog says
X = np.vstack([raw, np.ones_like(raw)]).T
alpha, beta = np.linalg.lstsq(X, priced, rcond=None)[0]
print(f"alpha={alpha:.3f}  beta={beta:.3f}")
print("re-priced:", (alpha * raw + beta).round(1))
print("target   :", priced)


## 5. Runnable training on synthetic data


In [ ]:
# Train a tiny XGBoost regressor end-to-end on fake structured data.
import numpy as np
import xgboost as xgb

rng = np.random.default_rng(0)
N = 400
X = rng.normal(size=(N, 10))                       # 10 features
y = (2*X[:, 0] - X[:, 1]**2 + 0.5*X[:, 2]).reshape(-1) + rng.normal(scale=0.3, size=N)

dtr = xgb.DMatrix(X[:300], label=y[:300])
dte = xgb.DMatrix(X[300:], label=y[300:])
params = {"objective": "reg:squarederror", "max_depth": 4, "eta": 0.1, "verbosity": 0}
booster = xgb.train(params, dtr, num_boost_round=80,
                    evals=[(dte, "test")], verbose_eval=20)

preds = booster.predict(dte)
print("RMSE on held-out:", np.sqrt(((preds - y[300:])**2).mean()).round(4))


**Next:** notebook 05 derives every evaluation metric we use — precision, recall, F1, RMSE, MAE, MAPE, R² — with worked numbers.
